In [1]:
import pandas as pd
from mlxtend.frequent_patterns import apriori, association_rules
from mlxtend.preprocessing import TransactionEncoder
import numpy as np

In [ ]:
df = pd.read_csv("orders_products.csv")


In [ ]:

# Проверяем размер
print("Исходный размер:", df.shape)
popular_products = [
    "Banana",
    "Bag of Organic Bananas",
    "Organic Strawberries",
    "Organic Baby Spinach",
    "Organic Hass Avocado",
    "Organic Avocado",
    "Large Lemon",
    "Organic Raspberries",
    "Organic Whole Milk",
    "Strawberries",
    "Limes",
    "Organic Garlic",
    "Organic Zucchini",
    "Organic Yellow Onion",
    "Cucumber Kirby",
    "Organic Blueberries",
    "Organic Fuji Apple",
    "Apple Honeycrisp Organic",
    "Organic Lemon",
    "Seedless Red Grapes",
    "Sparkling Water Grapefruit",
    "Yellow Onions",
    "Organic Baby Carrots",
    "Organic Baby Arugula",
    "Organic Grape Tomatoes",
    "Honeycrisp Apple",
    "Organic Half & Half",
    "Organic Cucumber",
    "Organic Small Bunch Celery",
    "Organic Large Extra Fancy Fuji Apple",
    "Carrots",
    "Original Hummus",
    "Organic Gala Apples",
    "Fresh Cauliflower",
    "Michigan Organic Kale",
    "Organic Red Onion",
    "Organic Blackberries",
    "Organic Cilantro",
    "Spring Water",
    "Half & Half",
    "Asparagus",
    "100% Whole Wheat Bread",
    "Raspberries",
    "Organic Italian Parsley Bunch",
    "Organic Unsweetened Almond Milk",
    "Organic Tomato Cluster",
    "Organic Whole String Cheese",
    "Organic Red Bell Pepper",
    "Red Vine Tomato"
]

#df_filtered = df[df['product_name'].isin(popular_products)]
#print("Исходный фильтр размер:", df_filtered.shape)
df = df.sample(n=1_000_00, random_state=42)

def replace_unpopular_products(group):
    items = group['product_name'].tolist()
    new_items = []
    for item in items:
        if item in popular_products:
            new_items.append(item)
        else:
            # выбираем случайный популярный товар (можно с повторением)
            new_item = np.random.choice(popular_products)
            new_items.append(new_item)
    group['product_name'] = new_items
    return group

# Применяем по каждому заказу
df_modified = df.groupby('order_id').apply(replace_unpopular_products).reset_index(drop=True)
print(1)
def remove_duplicates(group):
    # оставляем только уникальные продукты
    group = group.drop_duplicates(subset=['product_name'])
    return group

df_final = df_modified.groupby('order_id').apply(remove_duplicates).reset_index(drop=True)
print(df_modified.shape)
  # фиксируем random_state для воспроизводимости


# Сохраняем уменьшенный CSV
df_final.to_csv("orders_products_sample.csv", index=False)

In [2]:
df = pd.read_csv("orders.csv")



In [3]:
product_counts = df['product_name'].value_counts()
frequent_products = product_counts[product_counts >= 150].index
df = df[df['product_name'].isin(frequent_products)]
print(df.shape)
print(df['product_name'].value_counts()/len(df) * 100)


(1299912, 2)
product_name
Banana                                  9.958982
Spring Water                            8.488267
Bag of Organic Bananas                  7.491276
Limes                                   7.394731
Yellow Onions                           6.595523
Organic Strawberries                    6.168956
Organic Baby Spinach                    3.131981
Organic Hass Avocado                    2.633409
Organic Avocado                         2.434549
Large Lemon                             2.013213
Organic Raspberries                     1.983673
Organic Whole Milk                      1.904590
Strawberries                            1.853125
Organic Garlic                          1.508102
Organic Zucchini                        1.480331
Organic Yellow Onion                    1.462484
Cucumber Kirby                          1.457483
Organic Blueberries                     1.318012
Organic Fuji Apple                      1.302550
Apple Honeycrisp Organic                1.2

In [4]:
basket = df.groupby('order_id')['product_name'].apply(list)


In [5]:
te = TransactionEncoder()
te_array = te.fit(basket).transform(basket)
basket = pd.DataFrame(te_array, columns=te.columns_)


In [10]:
frequent = apriori(basket, min_support=0.1, use_colnames=True)

print("Частых наборов:", len(frequent))
display(frequent.head())

Частых наборов: 53


,support,itemsets
0,0.486900,(Bag of Organic Bananas)
1,0.647290,(Banana)
2,0.130850,(Large Lemon)
3,0.480625,(Limes)
4,0.158235,(Organic Avocado)


In [11]:
rules = association_rules(frequent, metric="confidence", min_threshold=0.1)

rules = rules[['antecedents','consequents','support','confidence','lift']]
rules['antecedents'] = rules['antecedents'].apply(lambda x: ", ".join(list(x)))
rules['consequents'] = rules['consequents'].apply(lambda x: ", ".join(list(x)))

print("Количество правил:", len(rules))
display(rules)

Количество правил: 164


,antecedents,consequents,support,confidence,lift
0,Bag of Organic Bananas,Banana,0.327030,0.671657,1.037645
1,Banana,Bag of Organic Bananas,0.327030,0.505229,1.037645
2,Bag of Organic Bananas,Limes,0.243730,0.500575,1.041509
3,Limes,Bag of Organic Bananas,0.243730,0.507111,1.041509
4,Organic Baby Spinach,Bag of Organic Bananas,0.101700,0.499595,1.026073
...,...,...,...,...,...
159,"Banana, Limes","Spring Water, Bag of Organic Bananas",0.105365,0.325965,1.168334
160,Spring Water,"Bag of Organic Bananas, Banana, Limes",0.105365,0.190982,1.119541
161,Bag of Organic Bananas,"Spring Water, Banana, Limes",0.105365,0.216400,1.126700
162,Banana,"Spring Water, Bag of Organic Bananas, Limes",0.105365,0.162779,1.118331


In [12]:
def simplify(itemset):
    return tuple(sorted(list(itemset)))

rules['A'] = rules['antecedents'].apply(simplify)
rules['C'] = rules['consequents'].apply(simplify)

# Для удаления дублей считаем объединённое множество
rules['union'] = rules.apply(lambda x: tuple(sorted(set(x['A']) | set(x['C']))), axis=1)

# Убираем дубли: одно union → одно правило с максимальной lift
rules_cleaned = rules.sort_values('lift', ascending=False).drop_duplicates('union')

# Чистим временные колонки
rules_cleaned = rules_cleaned.drop(columns=['A','C','union'])



print("Количество правил после очистки:", len(rules_cleaned))


Количество правил после очистки: 39


In [13]:
# Объединяем antecedents и consequents в один столбец
def merge_rule(row):
    merged = sorted(list(set(row['antecedents'].split(",")) | set(row['consequents'].split(","))))
    return ", ".join(merged)

rules_cleaned["rule_combined"] = rules_cleaned.apply(merge_rule, axis=1)

# Оставим только один столбец, если нужны только объединённые правила
rules_final = rules_cleaned.drop(columns=['antecedents','consequents'])

print("Количество уникальных объединённых правил:", len(rules_final))
display(rules_final)


Количество уникальных объединённых правил: 39


,support,confidence,lift,rule_combined
157,0.105365,0.322188,1.171740,"Banana, Limes, Bag of Organic Bananas, Sprin..."
143,0.127980,0.298544,1.085754,"Limes, Spring Water, Yellow Onions"
47,0.170590,0.354934,1.085325,"Banana, Bag of Organic Bananas, Limes"
126,0.171755,0.465026,1.084786,"Banana, Spring Water, Yellow Onions"
53,0.142075,0.354342,1.083514,"Banana, Bag of Organic Bananas, Organic Straw..."
58,0.194690,0.399856,1.082609,"Banana, Bag of Organic Bananas, Spring Water"
107,0.192065,0.399615,1.081956,"Banana, Limes, Spring Water"
79,0.112080,0.519840,1.081591,"Yellow Onions, Bag of Organic Bananas, Limes"
109,0.149390,0.519563,1.081016,"Yellow Onions, Banana, Limes"
69,0.105160,0.215979,1.081001,"Organic Strawberries, Bag of Organic Bananas,..."


In [24]:
def get_recommendations(item, metric, rules, top_n=5):
    """
    Возвращает рекомендации для товара item на основе правил ассоциаций.

    Параметры:
        item (str): товар, для которого ищутся рекомендации
        metric (str): метрика сортировки ('confidence', 'lift', 'support')
        rules (DataFrame): таблица правил
        top_n (int): количество рекомендаций
    
    Возвращает:
        DataFrame: top_n рекомендаций
    """

    # Проверяем корректность метрики
    allowed = ["support", "confidence", "lift"]
    if metric not in allowed:
        raise ValueError(f"metric должен быть одним из: {allowed}")

    df = rules.copy()

    # Разбиваем строку "Banana, Limes" → ["Banana", "Limes"]
    df["items_list"] = df["rule_combined"].apply(lambda x: [i.strip() for i in x.split(",")])

    # Оставляем только те строки, где item присутствует
    df = df[df["items_list"].apply(lambda x: item in x)]

    if df.empty:
        print(f"Нет рекомендаций для товара: {item}")
        return pd.DataFrame()

    # Рекомендации = все товары кроме item
    df["recommendation"] = df["items_list"].apply(
        lambda lst: [p for p in lst if p != item]
    )

    # Каждая строка может содержать несколько рекомендаций → распаковываем
    df = df.explode("recommendation")

    # Удаляем строки-пустышки
    df = df[df["recommendation"].notna()]

    # Сортируем по метрике
    df = df.sort_values(metric, ascending=False)

    return df[["recommendation", "support", "confidence", "lift"]].head(top_n).reset_index(drop=True)

In [25]:
get_recommendations("Banana", "lift", rules_final, top_n=5)

,recommendation,support,confidence,lift
0,Limes,0.105365,0.322188,1.171740
1,Bag of Organic Bananas,0.105365,0.322188,1.171740
2,Spring Water,0.105365,0.322188,1.171740
3,Bag of Organic Bananas,0.170590,0.354934,1.085325
4,Limes,0.170590,0.354934,1.085325


In [ ]:
get_recommendations("Banana", "lift", rules_final, top_n=5)